# Figure 2 · In-context adaptation at the training horizon

Frozen 8M endpoints on one task roster per environment, one row of three success views on the main text's benchmarks (Dark Key-to-Door, CountRecall and MazeRunner; the passive T-Maze in the appendix). Column 1: Dark Key-to-Door door success by attempt index (among finished attempts; the panel stops where a failure can still finish inside the 500-call budget; bands on the method and its carry control, every curve named at its last attempt). Column 2: CountRecall cumulative exact accuracy by scored query (the one-stream cells). Column 3: MazeRunner 15 × 15, the fraction of maps that reach goal k of the ordered sequence (a map reaches goal k when its episode completed at least k goals within the 500-step budget). Write boundaries are marked. The cumulative doors, doors per 100 calls and tasks-reaching-attempt views, the CountRecall accuracy per C32 block, the MazeRunner intervention view (retained, summary-cleared, goal-cleared), the summary-cleared traces and every T-Maze view (interventions, success bars, cumulative goals, success by cue side) are saved as standalone panels for the appendix, not drawn in the composite.

Rosters follow one rule: the confirmation split when every paper cell has records there, otherwise the split on which the most paper cells have records, labelled *provisional* when it is not the paper roster. A cell without records on the chosen split is named in the legend as pending and is never filled from another roster, so the competing methods always share one panel; nothing is written on a chart about it. Every cell reads saved records only; the estimators and the plotting code are in this notebook (the cell tagged `definitions`), the shared plumbing (record paths, the roster rule, the palette, the save step) in [figures_common.py](figures_common.py).

In [ ]:
"""Locate the repository, the study roots and the output folders.

Environment overrides, for a copy of the records elsewhere:
  REASONED_ICRL_OUTPUTS   study roots (default <repo>/outputs)
  FIGURE_OUTPUT_ROOT      where the figure is written (default notebooks/outputs/figures);
                          its tables and notes go to the sibling data/ folder
"""
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
sys.path.insert(0, str(REPO / "notebooks"))

from figures_common import default_layout, markdown_preview  # noqa: E402

OUTPUTS = Path(os.environ.get("REASONED_ICRL_OUTPUTS", REPO / "outputs"))
FIGURES_DIR = Path(
    os.environ.get("FIGURE_OUTPUT_ROOT", REPO / "notebooks/outputs/figures")
)
DATA_DIR = FIGURES_DIR.parent / "data"
layout = default_layout(REPO, outputs=OUTPUTS)
print(f"records under {OUTPUTS}\nfigures to  {FIGURES_DIR}\ntables to   {DATA_DIR}")

In [ ]:
from collections import defaultdict
from collections.abc import Sequence
from dataclasses import dataclass

import numpy as np
from figures_common import (
    DRAWN_CELLS,
    ENVIRONMENTS,
    FULL_WIDTH,
    GRID,
    INK_FAINT,
    INK_MUTED,
    METHOD_CONDITION,
    PANEL,
    SEEDS,
    SEGMENT,
    SUMMARY_CLEARED,
    BuiltFigure,
    CellStyle,
    band,
    choose_split,
    composite_legend,
    legend_handles,
    load_cell,
    paper_style,
    pending_handles,
    secondary_panel,
    standalone,
    style_axes,
)

from reasoned_icrl.analysis.statistics import (
    _exact_estimate,
    attempt_curves,
    count_recall_query_curves,
    cumulative_curves,
)
from reasoned_icrl.analysis.tier import budget_attempt_cutoff, supported_attempt_cutoff

COUNT_RECALL_QUERIES = 103
"""Scored queries of the CountRecall Medium stream (104 observations, one reset)."""
COUNT_RECALL_BLOCKS = ((1, 32), (33, 64), (65, 96), (97, 103))
"""The four C32 blocks of the stream: writes precede queries 33, 65, 97."""
FIGURE2_CELLS = DRAWN_CELLS
"""Every cell Figure 2 draws: the RSM family
(RSM-O, the method, and RSM-R, its rewrite ablation), the memoryless control and the four comparators, Memo with
fixed segments among them."""

TMAZE_HISTORIES = ("retained", "summary-cleared", "current-token")
"""The T-Maze intervention view: the retained panel beside the two declared
interventions (study protocol, section 11)."""
HISTORY_MARKS = {
    "retained": dict(marker="o", label="retained"),
    "summary-cleared": dict(marker="x", label="summary cleared"),
    "current-token": dict(marker="+", label="current token"),
    "goal-cleared": dict(marker="+", label="goal cleared"),
}
MAZERUNNER_HISTORIES = ("retained", "summary-cleared", "goal-cleared")
"""The MazeRunner intervention view: the retained panel beside the two
histories the evaluator declares for it (summary-cleared: the summary reset at
every boundary; goal-cleared: the record cleared at every goal)."""
END_LABELS = {
    "raw_summary_residual": "RSM-R",
    "raw_summary": "RSM-O",
    "raw_segment": "w/o memory",
    "full_context": "full history",
    "full_gru": "GRU",
    "memo": "Memo",
    "memo_fixed": "Memo fixed",
}
"""End-of-curve labels of the Key-to-Door attempt panel (the six curves converge at parity, so each is named at its last point and only the method and its carry control carry an interval band)."""


def _end_labels(ax, points, *, x, gap, fontsize=8.5):
    """Name each curve at its last point; labels are pushed apart vertically
    by at least ``gap`` (data units) so none overlap, with a thin leader from
    the curve's end when a label had to move."""
    placed = []
    for y, label, color in sorted(points, key=lambda point: point[0]):
        y_label = y
        if placed and y_label - placed[-1] < gap:
            y_label = placed[-1] + gap
        placed.append(y_label)
        ax.annotate(
            label,
            (x, y),
            xytext=(x + 0.25, y_label),
            textcoords="data",
            fontsize=fontsize,
            color=color,
            ha="left",
            va="center",
            annotation_clip=False,
            arrowprops={
                "arrowstyle": "-",
                "color": color,
                "linewidth": 0.5,
                "alpha": 0.9,
                "shrinkA": 1.0,
                "shrinkB": 1.5,
            }
            if abs(y_label - y) > 1e-9
            else None,
        )


TMAZE_TICKS = {
    "raw_summary": "RSM-O",
    "raw_segment": "w/o\nmemory",
    "raw_summary_residual": "RSM-R",
    "full_context": "full\nhistory",
    "full_gru": "GRU",
    "memo": "Memo",
    "memo_fixed": "Memo\nfixed",
}
"""Tick labels of the T-Maze column, one cell per tick."""


@dataclass(frozen=True)
class Figure2Options:
    seeds: Sequence[int] = SEEDS
    samples: int = 2000
    resampling_seed: int = 0
    split_preference: Sequence[str] = ("confirmation", "development")
    cells: Sequence[CellStyle] = FIGURE2_CELLS
    method: str = METHOD_CONDITION
    """The paper's RSM: the cell whose summary-cleared trace is drawn."""
    difference_omits: Sequence[str] = ("raw_segment",)
    """Cells the paired-difference panel leaves out: the memoryless control's
    gap is an order of magnitude larger than the carriers' and would flatten
    them. The legend names it."""
    attempt_dodge: float = 0.09
    """Attempts between the drawn positions of neighbouring cells in the
    success-by-attempt panel; 0 draws every cell on its index."""
    attempt_bands: Sequence[str] = (METHOD_CONDITION, "raw_segment")
    """Cells whose 95 % band the Key-to-Door attempt panel fills; the others
    are drawn as lines and named at their last point."""
    end_labels: bool = True
    """Name every curve of the attempt panel at its last attempt."""
    keydoor_grid_step: int = 10
    keydoor_window: int = 50
    environments: Sequence[str] = ("keydoor", "countrecall", "mazerunner")
    """The main text's columns:
    Key-to-Door, CountRecall and MazeRunner."""
    appendix_environments: Sequence[str] = ("tmaze",)
    """Environments whose panels are built standalone for the appendix only
    (the passive T-Maze) when their records exist."""
    figsize: tuple[float, float] = (FULL_WIDTH, 3.8)
    panel_size: tuple[float, float] = PANEL


# ---------------------------------------------------------------- estimators
# The uncertainty procedure is the study's own: a joint seed/task bootstrap of the
# (seed, task, rollout) cell values, 95 % interval, conditional on the trained seeds.


def _by_cell(events):
    grouped = defaultdict(list)
    for e in events:
        grouped[(e.training_seed, e.task_id, e.rollout_seed)].append(e)
    return grouped


def _window_row(condition, first, last, values, *, samples, rng):
    estimate, lower, upper, tasks, seeds, per_seed = _exact_estimate(
        values, samples=samples, confidence=0.95, rng=rng
    )
    return {
        "condition": condition,
        "first": first,
        "last": last,
        "centre": (first + last) / 2,
        "estimate": estimate,
        "lower": lower,
        "upper": upper,
        "tasks": tasks,
        "seeds": len(seeds),
        "per_seed": {str(s): round(v, 6) for s, v in per_seed.items()},
    }


def keydoor_window_rates(
    events, *, outer_length, width=50, per_calls=100, samples=2000, seed=0
):
    """Completed doors per ``per_calls`` charged calls in fixed ``width``-call windows.
    A door counts at the call its attempt ended, so resets are paid inside the clock,
    and every task contributes to every window."""
    rng = np.random.default_rng(seed)
    rows = []
    for condition in sorted({e.condition for e in events}):
        cells = _by_cell([e for e in events if e.condition == condition])
        for first in range(1, outer_length + 1, width):
            last = min(first + width - 1, outer_length)
            values = {
                cell: sum(
                    e.rate
                    for e in group
                    if e.end_step is not None and first <= e.end_step <= last
                )
                * per_calls
                / (last - first + 1)
                for cell, group in cells.items()
            }
            rows.append(
                _window_row(condition, first, last, values, samples=samples, rng=rng)
            )
    return rows


def countrecall_block_accuracy(
    events, *, blocks=COUNT_RECALL_BLOCKS, samples=2000, seed=0
):
    """Exact accuracy of the queries inside each block, one value per stream."""
    rng = np.random.default_rng(seed)
    rows = []
    for condition in sorted({e.condition for e in events}):
        cells = _by_cell([e for e in events if e.condition == condition])
        for first, last in blocks:
            values = {
                cell: float(
                    np.mean(
                        [e.numerator for e in group if first <= e.event_index <= last]
                    )
                )
                for cell, group in cells.items()
            }
            rows.append(
                _window_row(condition, first, last, values, samples=samples, rng=rng)
            )
    return rows


def keydoor_attempt_view(events, contract, *, samples=2000, seed=0):
    """Door success by attempt on finished attempts, with the risk set, and the index
    the panel stops at: the earlier of the 50 %-risk-set cut and the budget bound
    (past it every failure is censored by the outer budget, so the rate among
    finished attempts is 1 by construction)."""
    points = attempt_curves(events, samples=samples, seed=seed)
    cutoff = min(supported_attempt_cutoff(points), budget_attempt_cutoff(contract))
    rows = [
        {
            "condition": p.condition,
            "attempt": p.event_index,
            "estimate": p.estimate,
            "lower": p.lower,
            "upper": p.upper,
            "risk_set": p.risk_set,
            "shown": p.event_index <= cutoff,
        }
        for p in sorted(points, key=lambda p: (p.condition, p.event_index))
    ]
    return rows, cutoff


def keydoor_attempt_differences(events, *, method, cutoff, samples=2000, seed=0):
    """``method - cell`` at each attempt index, paired on the tasks where both
    finished that attempt.

    The success curves of the carriers lie within a few points of one another,
    so the absolute panel cannot show how far apart they are. This is the same
    reading as a paired contrast: for every attempt index, the units (seed,
    task, rollout) whose attempt finished under *both* cells, the difference
    of their outcomes, and the study's joint seed/task bootstrap over those
    units. A unit censored on either side is dropped, and the row says how
    many units are left."""
    rng = np.random.default_rng(seed)
    finished = [e for e in events if e.kind == "attempt" and e.complete]
    by_condition = {}
    for e in finished:
        by_condition.setdefault(e.condition, {}).setdefault(e.event_index, {})[
            (e.training_seed, e.task_id, e.rollout_seed)
        ] = float(e.numerator)
    rows = []
    mine = by_condition.get(method, {})
    for condition in sorted(by_condition):
        if condition == method:
            continue
        theirs = by_condition[condition]
        for index in sorted(set(mine) & set(theirs)):
            if index > cutoff:
                continue
            shared = set(mine[index]) & set(theirs[index])
            if not shared:
                continue
            values = {unit: mine[index][unit] - theirs[index][unit] for unit in shared}
            row = _window_row(condition, index, index, values, samples=samples, rng=rng)
            row["attempt"] = index
            row["units"] = len(shared)
            rows.append(row)
    return rows


def _cumulative_rows(points, split, axis):
    return [
        {
            "condition": p.condition,
            "split": split,
            axis: p.step,
            "estimate": p.estimate,
            "lower": p.lower,
            "upper": p.upper,
            "tasks": p.tasks,
            "seeds": p.seeds,
        }
        for p in sorted(points, key=lambda p: (p.condition, p.step))
    ]


# ------------------------------------------------------------------ compute
# Everything a column needs is computed once; the same data is then drawn into the
# composite and into each standalone panel.


def compute_keydoor_calls(selection, contract, layout, options, sources):
    outer = int(contract.environment.outer_length)
    grid = tuple(range(options.keydoor_grid_step, outer + 1, options.keydoor_grid_step))
    present = [c for c in options.cells if c.condition in selection.present]
    data = {
        "outer": outer,
        "present": present,
        "cumulative": (),
        "windows": [],
        "cleared": None,
    }
    if present:
        events = [e for c in present for e in selection.panels[c.condition]]
        data["cumulative"] = cumulative_curves(
            events, grid=grid, samples=options.samples, seed=options.resampling_seed
        )
        data["windows"] = keydoor_window_rates(
            events,
            outer_length=outer,
            width=options.keydoor_window,
            samples=options.samples,
            seed=options.resampling_seed,
        )
        cleared = load_cell(
            layout,
            "keydoor",
            options.method,
            selection.split,
            "summary-cleared",
            contract=contract,
            seeds=options.seeds,
            sources=sources,
        )
        if cleared is not None:
            data["cleared"] = sorted(
                cumulative_curves(
                    cleared,
                    grid=grid,
                    samples=options.samples,
                    seed=options.resampling_seed,
                ),
                key=lambda p: p.step,
            )
    return data


def compute_keydoor_attempts(selection, contract, options):
    present = [c for c in options.cells if c.condition in selection.present]
    data = {
        "present": present,
        "rows": [],
        "differences": [],
        "cutoff": 1,
        "largest": 0,
        "tail": 1,
    }
    if present:
        events = [e for c in present for e in selection.panels[c.condition]]
        rows, cutoff = keydoor_attempt_view(
            events, contract, samples=options.samples, seed=options.resampling_seed
        )
        data["differences"] = keydoor_attempt_differences(
            events,
            method=options.method,
            cutoff=cutoff,
            samples=options.samples,
            seed=options.resampling_seed,
        )
        largest = max(r["risk_set"] or 0 for r in rows)
        tail = max(
            (r["attempt"] for r in rows if (r["risk_set"] or 0) >= 0.01 * largest),
            default=cutoff,
        )
        data.update(rows=rows, cutoff=cutoff, largest=largest, tail=tail)
    return data


def compute_mazerunner(selection, contract, layout, options, sources):
    """The MazeRunner column at the trained size: goal fraction per cell (joint
    seed/map interval, per-seed values); the fraction of maps reaching goal k
    of the ordered sequence, k = 1 .. goals (the analogue of door success by
    attempt: a map reaches goal k when its episode completed at least k goals);
    the mean decisions taken; and the goal fraction under every declared
    history (the intervention view of the appendix)."""
    present = [c for c in options.cells if c.condition in selection.present]
    goals = int(contract.environment.goals)
    data = {
        "goals": goals,
        "budget": int(contract.environment.horizon),
        "present": present,
        "fraction": [],
        "reached": [],
        "decisions": [],
        "interventions": [],
    }
    if not present:
        return data
    rng = np.random.default_rng(options.resampling_seed)

    def fraction_row(condition, events, history):
        values = {
            cell: float(np.mean([e.numerator / e.denominator for e in group]))
            for cell, group in _by_cell(events).items()
        }
        row = _window_row(
            condition, goals, goals, values, samples=options.samples, rng=rng
        )
        row["history"] = history
        return row

    for cell in present:
        events = selection.panels[cell.condition]
        data["fraction"].append(fraction_row(cell.condition, events, "retained"))
        for k in range(1, goals + 1):
            values = {
                key: float(np.mean([1.0 if e.numerator >= k else 0.0 for e in group]))
                for key, group in _by_cell(events).items()
            }
            row = _window_row(
                cell.condition, k, k, values, samples=options.samples, rng=rng
            )
            row["goal"] = k
            data["reached"].append(row)
        data["decisions"].append(
            {
                "condition": cell.condition,
                "mean_decisions": float(np.mean([e.step for e in events])),
                **{
                    f"seed_{seed}": float(
                        np.mean([e.step for e in events if e.training_seed == seed])
                    )
                    for seed in options.seeds
                    if any(e.training_seed == seed for e in events)
                },
            }
        )
        for history in MAZERUNNER_HISTORIES:
            panel = (
                events
                if history == "retained"
                else load_cell(
                    layout,
                    "mazerunner",
                    cell.condition,
                    selection.split,
                    history,
                    contract=contract,
                    seeds=options.seeds,
                    sources=sources,
                )
            )
            if panel is not None:
                data["interventions"].append(
                    fraction_row(cell.condition, panel, history)
                )
    return data


def compute_countrecall(selection, contract, layout, options, sources):
    present = [c for c in options.cells if c.condition in selection.present]
    data = {
        "present": present,
        "curves": (),
        "blocks": [],
        "cleared_curve": None,
        "cleared_blocks": None,
    }
    if present:
        events = [e for c in present for e in selection.panels[c.condition]]
        data["curves"] = count_recall_query_curves(
            events,
            measure="cumulative_accuracy",
            samples=options.samples,
            seed=options.resampling_seed,
        )
        data["blocks"] = countrecall_block_accuracy(
            events, samples=options.samples, seed=options.resampling_seed
        )
        cleared = load_cell(
            layout,
            "countrecall",
            options.method,
            selection.split,
            "summary-cleared",
            contract=contract,
            seeds=options.seeds,
            sources=sources,
        )
        if cleared is not None:
            data["cleared_curve"] = sorted(
                count_recall_query_curves(
                    cleared,
                    measure="cumulative_accuracy",
                    samples=options.samples,
                    seed=options.resampling_seed,
                ),
                key=lambda p: p.step,
            )
            data["cleared_blocks"] = countrecall_block_accuracy(
                cleared, samples=options.samples, seed=options.resampling_seed
            )
    return data


def compute_tmaze(selection, contract, layout, options, sources):
    """The T-Maze column at the trained corridor: greedy success per cell (joint
    seed/task interval, per-seed values); cumulative goals reached over the
    roster's episodes (one per task, roster order) per seed; success by cue
    side per seed from the evaluator's secondary summaries, with the roster's
    cue balance (the success of a policy that always turns up, the declared
    cue-blind reference, and of one that always turns down); and the same
    success for every declared history (the intervention view of the appendix)."""
    present = [c for c in options.cells if c.condition in selection.present]
    corridor = int(contract.environment.size)
    data = {
        "corridor": corridor,
        "present": present,
        "success": [],
        "cumulative": {},
        "interventions": [],
        "cue_side": [],
        "cue_blind": None,
        "cue_up_fraction": None,
    }
    if not present:
        return data
    rng = np.random.default_rng(options.resampling_seed)

    def success_row(condition, events, history):
        values = {
            cell: float(np.mean([e.numerator for e in group]))
            for cell, group in _by_cell(events).items()
        }
        row = _window_row(
            condition, corridor, corridor, values, samples=options.samples, rng=rng
        )
        row["history"] = history
        return row

    for cell in present:
        events = selection.panels[cell.condition]
        data["success"].append(success_row(cell.condition, events, "retained"))
        per_seed = {}
        for seed in options.seeds:
            mine = sorted(
                (e for e in events if e.training_seed == seed),
                key=lambda e: (e.task_id, e.rollout_seed),
            )
            if mine:
                per_seed[seed] = np.cumsum([e.numerator for e in mine]).tolist()
        if per_seed:
            length = min(len(v) for v in per_seed.values())
            table = np.asarray([v[:length] for v in per_seed.values()], dtype=float)
            data["cumulative"][cell.condition] = {
                "episodes": list(range(1, length + 1)),
                "per_seed": {s: v[:length] for s, v in per_seed.items()},
                "mean": table.mean(axis=0).tolist(),
            }
        for history in TMAZE_HISTORIES:
            panel = (
                events
                if history == "retained"
                else load_cell(
                    layout,
                    "tmaze",
                    cell.condition,
                    selection.split,
                    history,
                    contract=contract,
                    seeds=options.seeds,
                    sources=sources,
                )
            )
            if panel is not None:
                data["interventions"].append(
                    success_row(cell.condition, panel, history)
                )
        for seed in options.seeds:
            secondary = secondary_panel(
                layout,
                "tmaze",
                cell.condition,
                seed,
                selection.split,
                "retained",
                sources=sources,
            )
            if secondary is None:
                continue
            data["cue_side"].append(
                {
                    "condition": cell.condition,
                    "seed": seed,
                    "success": secondary.get("goal_success"),
                    "success_cue_up": secondary.get("success_cue_up"),
                    "success_cue_down": secondary.get("success_cue_down"),
                    "tasks_cue_up": secondary.get("tasks_cue_up"),
                    "tasks_cue_down": secondary.get("tasks_cue_down"),
                    "forward_moves_mean": secondary.get("forward_moves_mean"),
                }
            )
            if data["cue_up_fraction"] is None and secondary.get("tasks"):
                data["cue_up_fraction"] = float(
                    secondary["tasks_cue_up"] / secondary["tasks"]
                )
                data["cue_blind"] = data["cue_up_fraction"]
    return data


# ------------------------------------------------------------------- drawing
# One function per panel; each draws into whatever axes it is given. A panel
# carries the environment's name as its title and nothing about missing cells:
# those are legend entries (figures_common.pending_handles).


def draw_cumulative_doors(ax, data, selection, options, *, cleared=False):
    """Cumulative completed doors against charged calls; the method's
    summary-cleared trace only when ``cleared`` (the appendix version)."""
    for cell in data["present"]:
        pts = sorted(
            (p for p in data["cumulative"] if p.condition == cell.condition),
            key=lambda p: p.step,
        )
        x = [0.0, *(float(p.step) for p in pts)]
        band(
            ax, x, [0.0, *(p.lower for p in pts)], [0.0, *(p.upper for p in pts)], cell
        )
        ax.plot(x, [0.0, *(p.estimate for p in pts)], **cell.line())
    if cleared and data["cleared"]:
        pts = data["cleared"]
        ax.plot(
            [0.0, *(float(p.step) for p in pts)],
            [0.0, *(p.estimate for p in pts)],
            **SUMMARY_CLEARED.markers(3.8, markevery=5, markeredgewidth=0.7),
        )
    for boundary in range(SEGMENT, data["outer"], SEGMENT):  # lazy C32 writes
        ax.axvline(boundary, color=GRID, linewidth=0.5, zorder=0)
    ax.set_xlim(0, data["outer"])
    ax.set_xticks(list(range(0, data["outer"] + 1, 100)))
    ax.set_ylim(0, 37)
    ax.set_xlabel("Charged environment calls")
    ax.set_ylabel("Cumulative doors per task")
    ax.set_title(selection.title(ENVIRONMENTS["keydoor"].title))
    style_axes(ax)


def draw_doors_per_100_calls(ax, data, selection, options):
    for cell in data["present"]:
        rows = [w for w in data["windows"] if w["condition"] == cell.condition]
        band(
            ax,
            [w["centre"] for w in rows],
            [w["lower"] for w in rows],
            [w["upper"] for w in rows],
            cell,
        )
        ax.plot(
            [w["centre"] for w in rows], [w["estimate"] for w in rows], **cell.markers()
        )
    for boundary in range(SEGMENT, data["outer"], SEGMENT):
        ax.axvline(boundary, color=GRID, linewidth=0.5, zorder=0)
    ax.set_xlim(0, data["outer"])
    ax.set_xticks(list(range(0, data["outer"] + 1, 100)))
    ax.set_ylim(0, 12)
    ax.set_xlabel("Charged environment calls")
    ax.set_ylabel(f"Doors per 100 calls ({options.keydoor_window}-call bins)")
    ax.set_title(selection.title(ENVIRONMENTS["keydoor"].title))
    style_axes(ax)


def draw_success_by_attempt(ax, data, options, selection):
    """Door success among finished attempts by attempt index. The cells are
    offset by a fraction of an attempt around their index: the carriers finish within a few points of one another, and markers
    drawn on the same x hide each other. The offset is a drawing device — the
    x of every point is its integer attempt index."""
    cutoff = data["cutoff"]
    ends = []
    width = options.attempt_dodge
    count = max(len(data["present"]) - 1, 1)
    for position, cell in enumerate(data["present"]):
        dodge = width * (position - count / 2) if width else 0.0
        shown = [
            r for r in data["rows"] if r["condition"] == cell.condition and r["shown"]
        ]
        x = [r["attempt"] + dodge for r in shown]
        if cell.condition in options.attempt_bands:
            band(
                ax,
                x,
                [r["lower"] for r in shown],
                [r["upper"] for r in shown],
                cell,
            )
        ax.plot(x, [r["estimate"] for r in shown], **cell.markers())
        if shown:
            last = max(shown, key=lambda r: r["attempt"])
            ends.append(
                (
                    last["estimate"],
                    END_LABELS.get(cell.condition, cell.label),
                    cell.color,
                )
            )
    if options.end_labels and ends:
        _end_labels(ax, ends, x=cutoff, gap=0.045)
        ax.set_xlim(0.5, cutoff + 3.4)
    else:
        ax.set_xlim(0.5, cutoff + 0.5)
    ax.set_xticks(list(range(1, cutoff + 1)))
    ax.set_ylim(0, 1.0)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_xlabel("Attempt index")
    ax.set_ylabel("Door success (finished attempts)")
    ax.set_title(selection.title(ENVIRONMENTS["keydoor"].title))
    style_axes(ax)


def draw_attempt_differences(ax, data, options):
    """The same attempts read as paired differences from the method: where the
    carriers' curves lie on top of one another, this panel says by how much
    and with what uncertainty. Zero is the method; a cell below it finished
    fewer attempts than the method on the same tasks. The memoryless control
    is left out — it sits an order of magnitude below and would flatten the
    carriers again — and the legend says so."""
    cutoff = data["cutoff"]
    drawn = [
        cell
        for cell in data["present"]
        if cell.condition != options.method
        and cell.condition not in options.difference_omits
    ]
    ax.axhline(0, color=INK_MUTED, linewidth=0.8, zorder=1)
    span = 0.05
    for cell in drawn:
        mine = sorted(
            (r for r in data["differences"] if r["condition"] == cell.condition),
            key=lambda r: r["attempt"],
        )
        if not mine:
            continue
        x = [r["attempt"] for r in mine]
        # Five intervals share this panel: a lighter fill keeps the curves
        # readable through them.
        band(
            ax,
            x,
            [r["lower"] for r in mine],
            [r["upper"] for r in mine],
            cell,
            alpha=0.07,
        )
        ax.plot(x, [r["estimate"] for r in mine], **cell.markers())
        span = max(
            span, *(abs(r["lower"]) for r in mine), *(abs(r["upper"]) for r in mine)
        )
    ax.set_xlim(0.5, cutoff + 0.5)
    ax.set_xticks(list(range(1, cutoff + 1)))
    ax.set_ylim(-1.15 * span, 1.15 * span)
    ax.set_xlabel("Attempt index")
    ax.set_ylabel(f"{END_LABELS.get(options.method, 'method')} - cell (door success)")
    ax.set_title(f"{ENVIRONMENTS['keydoor'].title} · paired by attempt")
    style_axes(ax)


def draw_tasks_reaching_attempt(ax, data, options):
    cutoff, largest, tail = data["cutoff"], data["largest"], data["tail"]
    for cell in data["present"]:
        reach = [
            r
            for r in data["rows"]
            if r["condition"] == cell.condition and r["attempt"] <= tail
        ]
        ax.plot(
            [r["attempt"] for r in reach],
            [(r["risk_set"] or 0) / largest for r in reach],
            **cell.line(),
        )
    if data["present"]:
        ax.axvline(
            cutoff + 0.5,
            color=INK_MUTED,
            linestyle=(0, (1, 2)),
            linewidth=0.8,
            zorder=1,
        )
        ax.text(
            cutoff + 0.45 * (tail - cutoff),
            0.9,
            f"success panel stops at attempt {cutoff}:\nlater failures are cut off\nby the 500-call budget",
            fontsize=8.5,
            color=INK_FAINT,
            va="top",
            ha="left",
            linespacing=1.05,
        )
        ax.set_xlim(0.5, tail + 0.5)
        ax.set_xticks([1, cutoff, *range(25, tail + 1, 25)])
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("Attempt index")
    ax.set_ylabel("Tasks reaching the attempt (fraction)")
    ax.set_title(ENVIRONMENTS["keydoor"].title)
    style_axes(ax)


def _blocks(ax, blocks, cell, *, marked):
    for b in blocks:
        span = [b["first"] - 0.5, b["last"] + 0.5]
        if marked:
            band(ax, span, [b["lower"]] * 2, [b["upper"]] * 2, cell)
        ax.plot(span, [b["estimate"]] * 2, **cell.line(solid_capstyle="butt"))
        ax.plot(
            b["centre"],
            b["estimate"],
            **cell.markers(3.6 if marked else 3.4, linestyle="none"),
        )


def _countrecall_axes(ax):
    for boundary in (32.5, 64.5, 96.5):  # writes before queries 33, 65, 97
        ax.axvline(boundary, color=GRID, linewidth=0.5, zorder=0)
    ax.set_xlim(0.5, COUNT_RECALL_QUERIES + 0.5)
    ax.set_xticks([1, 32, 64, 96])
    ax.set_ylim(0, 1.0)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_xlabel("Scored query index")
    style_axes(ax)


def draw_cumulative_accuracy(ax, data, selection, options, *, cleared=False):
    for cell in data["present"]:
        pts = sorted(
            (p for p in data["curves"] if p.condition == cell.condition),
            key=lambda p: p.step,
        )
        band(
            ax,
            [p.step for p in pts],
            [p.lower for p in pts],
            [p.upper for p in pts],
            cell,
        )
        ax.plot(
            [p.step for p in pts],
            [p.estimate for p in pts],
            **cell.markers(3.6, markevery=12),
        )
    if cleared and data["cleared_curve"]:
        pts = data["cleared_curve"]
        ax.plot(
            [p.step for p in pts],
            [p.estimate for p in pts],
            **SUMMARY_CLEARED.markers(3.8, markevery=8, markeredgewidth=0.7),
        )
    _countrecall_axes(ax)
    # The y-range follows the drawn curves: every
    # carrier sits near 1.0 and the control near 0.7, so the panel starts a
    # little below the lowest drawn interval instead of at zero.
    lowest = min(
        (
            p.lower
            for cell in data["present"]
            for p in data["curves"]
            if p.condition == cell.condition
        ),
        default=0.0,
    )
    if cleared and data["cleared_curve"]:
        lowest = min(lowest, min(p.estimate for p in data["cleared_curve"]))
    floor = float(max(0.0, np.floor((lowest - 0.03) * 20) / 20))
    ax.set_ylim(floor, 1.005)
    ax.set_yticks([t for t in np.arange(0.0, 1.0001, 0.1) if t >= floor - 1e-9])
    ax.set_ylabel("Cumulative exact accuracy")
    ax.set_title(selection.title(ENVIRONMENTS["countrecall"].title))


def draw_accuracy_per_block(ax, data, selection, options, *, cleared=False):
    for cell in data["present"]:
        _blocks(
            ax,
            [b for b in data["blocks"] if b["condition"] == cell.condition],
            cell,
            marked=True,
        )
    if cleared and data["cleared_blocks"]:
        _blocks(ax, data["cleared_blocks"], SUMMARY_CLEARED, marked=False)
    _countrecall_axes(ax)
    ax.set_ylabel("Accuracy per block")
    ax.set_title(selection.title(ENVIRONMENTS["countrecall"].title))


def draw_mazerunner_goals_reached(ax, data, selection, options):
    """The fraction of maps that reach goal k of the ordered sequence, by k
    (bands on the cells of ``attempt_bands``, every curve named at its last
    point): the MazeRunner form of door success by attempt."""
    goals = data["goals"]
    ends = []
    for cell in data["present"]:
        rows = sorted(
            (r for r in data["reached"] if r["condition"] == cell.condition),
            key=lambda r: r["goal"],
        )
        if not rows:
            continue
        if cell.condition in options.attempt_bands:
            band(
                ax,
                [r["goal"] for r in rows],
                [r["lower"] for r in rows],
                [r["upper"] for r in rows],
                cell,
            )
        ax.plot(
            [r["goal"] for r in rows], [r["estimate"] for r in rows], **cell.markers()
        )
        ends.append(
            (
                rows[-1]["estimate"],
                END_LABELS.get(cell.condition, cell.label),
                cell.color,
            )
        )
    if options.end_labels and ends:
        _end_labels(ax, ends, x=goals, gap=0.06)
        ax.set_xlim(0.5, goals + 1.6)
    else:
        ax.set_xlim(0.5, goals + 0.5)
    ax.set_xticks(list(range(1, goals + 1)))
    ax.set_ylim(0, 1.0)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_xlabel(f"Goal index k (of {goals}; {data['budget']}-step budget)")
    ax.set_ylabel("Maps reaching goal k")
    ax.set_title(selection.title(ENVIRONMENTS["mazerunner"].title))
    style_axes(ax)


def draw_mazerunner_interventions(ax, data, selection, options):
    """The intervention view: retained (the cell's marker), summary-cleared
    (cross) and goal-cleared (plus) goal fraction per cell, joined by a thin
    line."""
    offsets = {"retained": -0.2, "summary-cleared": 0.0, "goal-cleared": 0.2}
    present = data["present"]
    for x, cell in enumerate(present):
        mine = {
            r["history"]: r
            for r in data["interventions"]
            if r["condition"] == cell.condition
        }
        if not mine:
            continue
        xs = [x + offsets[h] for h in MAZERUNNER_HISTORIES if h in mine]
        ys = [mine[h]["estimate"] for h in MAZERUNNER_HISTORIES if h in mine]
        ax.plot(xs, ys, color=cell.color, linewidth=0.5, alpha=0.4, zorder=cell.zorder)
        for history, row in mine.items():
            mark = HISTORY_MARKS[history]
            ax.errorbar(
                x + offsets[history],
                row["estimate"],
                yerr=[
                    [row["estimate"] - row["lower"]],
                    [row["upper"] - row["estimate"]],
                ],
                color=cell.color,
                elinewidth=0.6,
                capsize=0,
                marker=mark["marker"],
                markersize=5.0 if history == "retained" else 5.8,
                markerfacecolor=cell.marker_face
                if history == "retained"
                else cell.color,
                markeredgewidth=0.8,
                linestyle="none",
                zorder=cell.zorder + 10,
            )
    ax.set_xlim(-0.6, max(len(present), 1) - 0.4)
    ax.set_xticks(range(len(present)))
    ax.set_xticklabels(
        [TMAZE_TICKS.get(c.condition, c.label) for c in present], fontsize=8.5
    )
    ax.set_ylim(0, 1.05)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_ylabel("Goal fraction by history")
    ax.set_xlabel("● retained    x summary cleared    + goal cleared")
    ax.set_title(selection.title(ENVIRONMENTS["mazerunner"].title))
    style_axes(ax)


def _tmaze_axes(
    ax, data, selection, options, *, view=None, cue_line=True, cue_label="inside"
):
    present = data["present"]
    ax.set_xlim(-0.6, max(len(present), 1) - 0.4)
    ax.set_xticks(range(len(present)))
    ax.set_xticklabels(
        [TMAZE_TICKS.get(c.condition, c.label) for c in present], fontsize=8.5
    )
    ax.set_ylim(0, 1.05)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    if cue_line and data["cue_blind"] is not None:
        ax.axhline(
            data["cue_blind"],
            color=INK_FAINT,
            linestyle=(0, (1, 2)),
            linewidth=0.7,
            zorder=0,
        )
        if cue_label == "inside":
            ax.text(
                ax.get_xlim()[1],
                data["cue_blind"] - 0.02,
                "cue-blind",
                color=INK_FAINT,
                fontsize=8.5,
                ha="right",
                va="top",
            )
        else:  # the bars fill the axes: the label sits just past the right edge
            ax.annotate(
                "cue-\nblind",
                (1.0, data["cue_blind"]),
                xycoords=("axes fraction", "data"),
                xytext=(2, 0),
                textcoords="offset points",
                color=INK_FAINT,
                fontsize=8.5,
                ha="left",
                va="center",
                annotation_clip=False,
            )
    title = ENVIRONMENTS["tmaze"].title
    title = selection.title(title) if view is None else f"{title} · {view}"
    ax.set_title(title)
    style_axes(ax)


def draw_tmaze_cumulative_goals(ax, data, selection, options):
    """Cumulative goals reached over the roster's episodes (one per task, in
    roster order): the seed mean as the line, the seeds as thin lines, so a
    seed at the cue-blind rate and a seed that solved the task separate; the
    band is the cue-blind reference, a fixed turn to either side (its slope is
    the fraction of tasks whose cue matches that side)."""
    curves = data["cumulative"]
    length = max((len(c["episodes"]) for c in curves.values()), default=0)
    if length and data["cue_up_fraction"] is not None:
        up = data["cue_up_fraction"]
        low, high = min(up, 1.0 - up), max(up, 1.0 - up)
        ax.fill_between(
            [0, length],
            [0, low * length],
            [0, high * length],
            color="#000000",
            alpha=0.06,
            linewidth=0,
            zorder=0,
        )
        ax.text(
            length * 0.99,
            low * length * 0.97,
            "cue-blind\n(fixed turn)",
            color=INK_FAINT,
            fontsize=8.5,
            ha="right",
            va="top",
        )
    for cell in data["present"]:
        curve = curves.get(cell.condition)
        if curve is None:
            continue
        for values in curve["per_seed"].values():
            ax.plot(
                curve["episodes"],
                values,
                color=cell.color,
                linewidth=0.45,
                alpha=0.35,
                linestyle=cell.linestyle,
                zorder=cell.zorder + 5,
            )
        ax.plot(curve["episodes"], curve["mean"], **cell.line())
    ax.set_xlim(0, max(length, 1))
    ax.set_ylim(0, max(length, 1) * 1.02)
    ax.set_xlabel("Episodes evaluated (one per task)")
    ax.set_ylabel(f"Cumulative goals, corridor {data['corridor']}")
    ax.set_title(selection.title(ENVIRONMENTS["tmaze"].title))
    style_axes(ax)


CUE_MARKS = {"up": ("^", -0.26), "down": ("v", 0.26)}
"""Marker and x-offset of the success by cue side."""


def draw_tmaze_success_by_cue(ax, data, selection, options):
    """Greedy success at the trained corridor per cell: all tasks (the cell's
    marker, joint seed/task interval) between the up-cue and down-cue success
    (triangles; seed means, the seeds as small markers behind them), so a
    policy that ignores the cue shows one side at 1 and the other at 0 while a
    policy that read it shows both at 1."""
    for x, cell in enumerate(data["present"]):
        (row,) = [r for r in data["success"] if r["condition"] == cell.condition]
        for value in row["per_seed"].values():
            ax.plot(
                x,
                value,
                marker=cell.marker,
                markersize=3.2,
                color=cell.color,
                alpha=0.45,
                linestyle="none",
                markerfacecolor=cell.marker_face,
                markeredgewidth=0.5,
                zorder=cell.zorder + 5,
            )
        ax.errorbar(
            x,
            row["estimate"],
            yerr=[[row["estimate"] - row["lower"]], [row["upper"] - row["estimate"]]],
            color=cell.color,
            elinewidth=0.8,
            capsize=0,
            marker=cell.marker,
            markersize=5.2,
            markerfacecolor=cell.marker_face,
            markeredgewidth=0.8,
            linestyle="none",
            zorder=cell.zorder + 10,
        )
        sides = [r for r in data["cue_side"] if r["condition"] == cell.condition]
        for side, (marker, dx) in CUE_MARKS.items():
            values = [
                r[f"success_cue_{side}"]
                for r in sides
                if r.get(f"success_cue_{side}") is not None
            ]
            if not values:
                continue
            for value in values:
                ax.plot(
                    x + dx,
                    value,
                    marker=marker,
                    markersize=3.4,
                    color=cell.color,
                    alpha=0.45,
                    linestyle="none",
                    markerfacecolor=cell.marker_face,
                    markeredgewidth=0.5,
                    zorder=cell.zorder + 5,
                )
            ax.plot(
                x + dx,
                float(np.mean(values)),
                marker=marker,
                markersize=5.4,
                color=cell.color,
                linestyle="none",
                markerfacecolor=cell.marker_face,
                markeredgewidth=0.8,
                zorder=cell.zorder + 10,
            )
    ax.set_ylabel(f"Greedy success, corridor {data['corridor']}")
    _tmaze_axes(ax, data, selection, options, view="by cue side")
    ax.set_xlabel("▲ up cue     ● all tasks     ▼ down cue")


def draw_tmaze_success_bars(ax, data, selection, options):
    """Greedy success at the trained corridor as one bar per cell (the joint
    seed/task estimate, its interval as the error bar, the seeds as small
    markers), in the cell's colour; a cell drawn hollow elsewhere is a hollow
    bar here."""
    for x, cell in enumerate(data["present"]):
        (row,) = [r for r in data["success"] if r["condition"] == cell.condition]
        hollow = cell.marker_face != cell.color
        ax.bar(
            x,
            row["estimate"],
            width=0.62,
            color=cell.color if not hollow else "white",
            edgecolor=cell.color,
            linewidth=0.9,
            hatch="////" if hollow else None,
            zorder=2,
        )
        ax.errorbar(
            x,
            row["estimate"],
            yerr=[[row["estimate"] - row["lower"]], [row["upper"] - row["estimate"]]],
            color=INK_MUTED if not hollow else cell.color,
            elinewidth=0.8,
            capsize=2,
            capthick=0.8,
            linestyle="none",
            zorder=4,
        )
        for value in row["per_seed"].values():
            ax.plot(
                x,
                value,
                marker=cell.marker,
                markersize=3.4,
                color=cell.color,
                linestyle="none",
                markerfacecolor="white",
                markeredgewidth=0.6,
                zorder=5,
            )
    ax.set_ylabel(f"Greedy success, corridor {data['corridor']}")
    _tmaze_axes(ax, data, selection, options, cue_label="outside")


def draw_tmaze_interventions(ax, data, selection, options):
    """The intervention view: retained (the cell's marker), summary-cleared
    (cross) and current-token (plus) success per cell, joined by a thin line, so
    what the junction decision is read from is visible (the T-Maze form of the
    cleared trace)."""
    offsets = {"retained": -0.2, "summary-cleared": 0.0, "current-token": 0.2}
    for x, cell in enumerate(data["present"]):
        mine = {
            r["history"]: r
            for r in data["interventions"]
            if r["condition"] == cell.condition
        }
        if not mine:
            continue
        xs = [x + offsets[h] for h in TMAZE_HISTORIES if h in mine]
        ys = [mine[h]["estimate"] for h in TMAZE_HISTORIES if h in mine]
        ax.plot(xs, ys, color=cell.color, linewidth=0.5, alpha=0.4, zorder=cell.zorder)
        for history, row in mine.items():
            mark = HISTORY_MARKS[history]
            ax.errorbar(
                x + offsets[history],
                row["estimate"],
                yerr=[
                    [row["estimate"] - row["lower"]],
                    [row["upper"] - row["estimate"]],
                ],
                color=cell.color,
                elinewidth=0.6,
                capsize=0,
                marker=mark["marker"],
                markersize=5.0 if history == "retained" else 5.8,
                markerfacecolor=cell.marker_face
                if history == "retained"
                else cell.color,
                markeredgewidth=0.8,
                linestyle="none",
                zorder=cell.zorder + 10,
            )
    ax.set_ylabel("Greedy success by history")
    _tmaze_axes(ax, data, selection, options, view="interventions")
    ax.set_xlabel("● retained    x summary cleared    + current token")


# --------------------------------------------------------------------- build


def _pending(selection, environment, options):
    """The cells without records on the column's roster, for the legend."""
    return [c for c in options.cells if c.condition in selection.missing]


def build_figure2(layout, options=None):
    """Figure 2 from the saved records under ``layout``: success views on the
    in-context axis (attempts, queries, goals) with write boundaries marked;
    every panel also standalone, the summary-cleared traces beside them."""
    options = options or Figure2Options()
    sources = []
    conditions = [c.condition for c in options.cells]
    keydoor_contract = layout.contract("keydoor")
    countrecall_contract = layout.contract("countrecall")
    mazerunner_contract = layout.contract("mazerunner")
    tmaze_contract = layout.contract("tmaze")
    selections = {}
    for environment, contract in (
        ("keydoor", keydoor_contract),
        ("countrecall", countrecall_contract),
        ("mazerunner", mazerunner_contract),
        ("tmaze", tmaze_contract),
    ):
        selections[environment] = choose_split(
            layout,
            environment,
            conditions,
            contract=contract,
            preference=options.split_preference,
            seeds=options.seeds,
            sources=sources,
        )
    keydoor, countrecall = selections["keydoor"], selections["countrecall"]
    mazerunner, tmaze = selections["mazerunner"], selections["tmaze"]
    calls = compute_keydoor_calls(keydoor, keydoor_contract, layout, options, sources)
    attempts = compute_keydoor_attempts(keydoor, keydoor_contract, options)
    queries = compute_countrecall(
        countrecall, countrecall_contract, layout, options, sources
    )
    runner = compute_mazerunner(
        mazerunner, mazerunner_contract, layout, options, sources
    )
    maze = compute_tmaze(tmaze, tmaze_contract, layout, options, sources)
    pending = {
        ENVIRONMENTS[e].title: _pending(selections[e], e, options)
        for e in options.environments
    }
    handles = legend_handles(options.cells) + pending_handles(
        pending, options.cells, reason="pending (no records on the roster)"
    )
    with paper_style():
        # One row, one panel per main-text environment (Key-to-Door, CountRecall and MazeRunner): door
        # success by attempt (bands on the method and its carry control, every
        # curve named at its end), cumulative exact accuracy by query, and the
        # fraction of maps reaching each goal of the sequence. The legend sits
        # in the figure margin above the panels (constrained layout).
        fig = plt.figure(figsize=options.figsize, layout="constrained")
        axes = fig.subplots(1, 3)
        draw_success_by_attempt(axes[0], attempts, options, keydoor)
        draw_cumulative_accuracy(axes[1], queries, countrecall, options)
        draw_mazerunner_goals_reached(axes[2], runner, mazerunner, options)
        composite_legend(fig, handles, ncol=3)
    size = options.panel_size

    def standalone_handles(selection, environment, present):
        return legend_handles(present) + pending_handles(
            {
                ENVIRONMENTS[environment].title: _pending(
                    selection, environment, options
                )
            },
            options.cells,
            reason="pending (no records on the roster)",
        )

    keydoor_handles = standalone_handles(keydoor, "keydoor", calls["present"])
    query_handles = standalone_handles(countrecall, "countrecall", queries["present"])
    runner_handles = standalone_handles(mazerunner, "mazerunner", runner["present"])
    tmaze_handles = standalone_handles(tmaze, "tmaze", maze["present"])
    cleared_handle = [SUMMARY_CLEARED.handle()]
    panels = {
        "keydoor_cumulative_doors": standalone(
            lambda ax: draw_cumulative_doors(ax, calls, keydoor, options),
            size=size,
            handles=keydoor_handles,
        ),
        "keydoor_success_by_attempt": standalone(
            lambda ax: draw_success_by_attempt(ax, attempts, options, keydoor),
            size=size,
            handles=keydoor_handles,
        ),
        "keydoor_doors_per_100_calls": standalone(
            lambda ax: draw_doors_per_100_calls(ax, calls, keydoor, options),
            size=size,
            handles=keydoor_handles,
        ),
        "keydoor_success_by_attempt_difference": standalone(
            lambda ax: draw_attempt_differences(ax, attempts, options),
            size=size,
            handles=[
                *legend_handles(
                    c
                    for c in attempts["present"]
                    if c.condition != options.method
                    and c.condition not in options.difference_omits
                ),
                *pending_handles(
                    {
                        "Not drawn": [
                            c
                            for c in attempts["present"]
                            if c.condition in options.difference_omits
                        ]
                    },
                    options.cells,
                    reason="(its gap would flatten the carriers)",
                ),
            ],
        ),
        "keydoor_tasks_reaching_attempt": standalone(
            lambda ax: draw_tasks_reaching_attempt(ax, attempts, options),
            size=size,
            handles=keydoor_handles,
        ),
        "countrecall_cumulative_accuracy": standalone(
            lambda ax: draw_cumulative_accuracy(ax, queries, countrecall, options),
            size=size,
            handles=query_handles,
        ),
        "countrecall_accuracy_per_block": standalone(
            lambda ax: draw_accuracy_per_block(ax, queries, countrecall, options),
            size=size,
            handles=query_handles,
        ),
        "mazerunner_goals_reached": standalone(
            lambda ax: draw_mazerunner_goals_reached(ax, runner, mazerunner, options),
            size=size,
            handles=runner_handles,
        ),
        "mazerunner_interventions": standalone(
            lambda ax: draw_mazerunner_interventions(ax, runner, mazerunner, options),
            size=size,
            handles=runner_handles,
        ),
        "tmaze_cumulative_goals": standalone(
            lambda ax: draw_tmaze_cumulative_goals(ax, maze, tmaze, options),
            size=size,
            handles=tmaze_handles,
        ),
        "tmaze_success_bars": standalone(
            lambda ax: draw_tmaze_success_bars(ax, maze, tmaze, options),
            size=size,
            handles=tmaze_handles,
        ),
        "tmaze_success_by_cue": standalone(
            lambda ax: draw_tmaze_success_by_cue(ax, maze, tmaze, options),
            size=size,
            handles=tmaze_handles,
        ),
        "tmaze_interventions": standalone(
            lambda ax: draw_tmaze_interventions(ax, maze, tmaze, options),
            size=size,
            handles=tmaze_handles,
        ),
    }
    if calls["cleared"]:
        panels["keydoor_cumulative_doors_summary_cleared"] = standalone(
            lambda ax: draw_cumulative_doors(ax, calls, keydoor, options, cleared=True),
            size=size,
            handles=keydoor_handles + cleared_handle,
        )
    if queries["cleared_curve"]:
        panels["countrecall_cumulative_accuracy_summary_cleared"] = standalone(
            lambda ax: draw_cumulative_accuracy(
                ax, queries, countrecall, options, cleared=True
            ),
            size=size,
            handles=query_handles + cleared_handle,
        )
        panels["countrecall_accuracy_per_block_summary_cleared"] = standalone(
            lambda ax: draw_accuracy_per_block(
                ax, queries, countrecall, options, cleared=True
            ),
            size=size,
            handles=query_handles + cleared_handle,
        )
    tables = {}
    if calls["present"]:
        tables["keydoor_cumulative"] = _cumulative_rows(
            calls["cumulative"], keydoor.split, "step"
        )
        tables["keydoor_windows"] = [
            {**w, "split": keydoor.split} for w in calls["windows"]
        ]
        tables["keydoor_attempts"] = [
            {**r, "split": keydoor.split} for r in attempts["rows"]
        ]
        tables["keydoor_attempt_differences"] = [
            {**r, "split": keydoor.split, "method": options.method}
            for r in attempts["differences"]
        ]
        if calls["cleared"]:
            tables["keydoor_summary_cleared"] = _cumulative_rows(
                calls["cleared"], keydoor.split, "step"
            )
    if queries["present"]:
        tables["countrecall_cumulative"] = _cumulative_rows(
            queries["curves"], countrecall.split, "query"
        )
        tables["countrecall_blocks"] = [
            {**b, "split": countrecall.split} for b in queries["blocks"]
        ]
        if queries["cleared_curve"]:
            tables["countrecall_summary_cleared"] = _cumulative_rows(
                queries["cleared_curve"], countrecall.split, "query"
            )
            tables["countrecall_summary_cleared_blocks"] = [
                {**b, "split": countrecall.split} for b in queries["cleared_blocks"]
            ]
    if runner["present"]:
        tables["mazerunner_goal_fraction"] = [
            {**r, "split": mazerunner.split} for r in runner["fraction"]
        ]
        tables["mazerunner_goals_reached"] = [
            {**r, "split": mazerunner.split} for r in runner["reached"]
        ]
        tables["mazerunner_decisions"] = [
            {**r, "split": mazerunner.split} for r in runner["decisions"]
        ]
        tables["mazerunner_interventions"] = [
            {**r, "split": mazerunner.split} for r in runner["interventions"]
        ]
    if maze["present"]:
        tables["tmaze_success"] = [{**r, "split": tmaze.split} for r in maze["success"]]
        tables["tmaze_cumulative"] = [
            {
                "condition": condition,
                "split": tmaze.split,
                "episode": episode,
                "mean": curve["mean"][index],
                **{
                    f"seed_{seed}": values[index]
                    for seed, values in curve["per_seed"].items()
                },
            }
            for condition, curve in maze["cumulative"].items()
            for index, episode in enumerate(curve["episodes"])
        ]
        tables["tmaze_interventions"] = [
            {**r, "split": tmaze.split} for r in maze["interventions"]
        ]
        tables["tmaze_cue_side"] = [
            {**r, "split": tmaze.split} for r in maze["cue_side"]
        ]
    notes = {
        "keydoor": keydoor.notes(),
        "countrecall": countrecall.notes(),
        "mazerunner": {**mazerunner.notes(), "goals": runner["goals"]},
        "tmaze": {
            **tmaze.notes(),
            "cue_blind": maze["cue_blind"],
            "cue_up_fraction": maze["cue_up_fraction"],
        },
        "attempt_cutoff": attempts["cutoff"],
        "attempt_dodge": options.attempt_dodge,
        "attempt_differences": "method - cell per attempt, paired on the tasks "
        "whose attempt finished under both cells; the panel omits "
        + ", ".join(options.difference_omits),
        "seeds": list(options.seeds),
        "samples": options.samples,
        "resampling_seed": options.resampling_seed,
        "checkpoint": "policy_epoch_999 (collection endpoint)",
        "interval": "95 % joint seed/task bootstrap, conditional on the trained seeds",
        "pending_in_legend": {k: [c.condition for c in v] for k, v in pending.items()},
        "composite": "1 row x 3 main-text columns (Key-to-Door door success by "
        "attempt, CountRecall cumulative exact accuracy, MazeRunner maps "
        "reaching each goal); the T-Maze and the other views standalone only "
        "(appendix)",
    }
    return BuiltFigure(
        "figure2_in_context_adaptation", fig, tables, notes, sources, panels
    )

In [ ]:
# Parameters: `cells` restricts the drawn cells (fixed order), `split_preference` is the
# roster rule, `method` the paper's RSM (its summary-cleared trace; the residual rewrite), `samples` the bootstrap draw count (2000 in the paper).
options = Figure2Options(
    samples=2000,
    resampling_seed=0,
    split_preference=("confirmation", "development"),
    cells=FIGURE2_CELLS,  # the main text's six cells; restrict with figures_common.select_cells([...])
)
figure2 = build_figure2(layout, options)
display(figure2.figure)

In [ ]:
# Which roster each column sits on and which cells are pending (named in the legend).
for environment in ("keydoor", "countrecall", "mazerunner", "tmaze"):
    print(environment, figure2.notes[environment])

In [ ]:
# The plotted data behind two panels (every table is saved as CSV below).
display(
    Markdown(markdown_preview(figure2.tables.get("keydoor_attempts", []), limit=10))
)
display(
    Markdown(
        markdown_preview(figure2.tables.get("mazerunner_goals_reached", []), limit=24)
    )
)

In [ ]:
# Save: the composite (PDF + PNG) and every panel (PNG) under FIGURES_DIR, one CSV
# per table and the notes under DATA_DIR.
for path in figure2.save(FIGURES_DIR, DATA_DIR):
    print(path.relative_to(REPO) if path.is_relative_to(REPO) else path)
figure2.close()